<a href="https://colab.research.google.com/github/anjalikrishnaak6-research/neuroimag_EEG/blob/main/classical_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

here we need to build a classical baseline model, using LDA, linear SVM, stratisified cross validation, balanced accuracy, AUC and permutation testing.

In [ ]:
!pip install scikit-learn

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import permutation_test_score

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
lda_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis())
])

In [ ]:
lda_bal_acc = cross_val_score(
    lda_pipeline,
    X_classical,
    y,
    cv=cv,
    scoring='balanced_accuracy'
)

print("LDA Balanced Accuracy: %.3f ± %.3f" %
      (lda_bal_acc.mean(), lda_bal_acc.std()))

In [ ]:
# AUC

lda_auc = cross_val_score(
    lda_pipeline,
    X_classical,
    y,
    cv=cv,
    scoring='roc_auc'
)

print("LDA AUC: %.3f ± %.3f" %
      (lda_auc.mean(), lda_auc.std()))

In [ ]:
# build linear SVM pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', probability=True))
])

In [ ]:
# evaluate SVM
svm_bal_acc = cross_val_score(
    svm_pipeline,
    X_classical,
    y,
    cv=cv,
    scoring='balanced_accuracy'
)

print("SVM Balanced Accuracy: %.3f ± %.3f" %
      (svm_bal_acc.mean(), svm_bal_acc.std()))

In [ ]:
svm_auc = cross_val_score(
    svm_pipeline,
    X_classical,
    y,
    cv=cv,
    scoring='roc_auc'
)

print("SVM AUC: %.3f ± %.3f" %
      (svm_auc.mean(), svm_auc.std()))

In [ ]:
# permutation testing LDA
score, perm_scores, pvalue = permutation_test_score(
    lda_pipeline,
    X_classical,
    y,
    cv=cv,
    scoring='balanced_accuracy',
    n_permutations=1000,
    random_state=42
)

print("LDA Permutation Test Score:", score)
print("p-value:", pvalue)

In [ ]:
# permutation testing SVM

score, perm_scores, pvalue = permutation_test_score(
    svm_pipeline,
    X_classical,
    y,
    cv=cv,
    scoring='balanced_accuracy',
    n_permutations=1000,
    random_state=42
)

print("SVM Permutation Test Score:", score)
print("p-value:", pvalue)

In [ ]:
# confusion matrix

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X_classical, y, test_size=0.2, stratify=y, random_state=42
)

lda_pipeline.fit(X_train, y_train)
y_pred = lda_pipeline.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)